# Why do we Need to Backtest and What Tests Should we Use for it?
## (1) The need for backtesting in risk
We want to know whether our risk forecasts behave as we claimed on the historical data, which better informs our risk decisions.

"The empiricist tradition in the philosophy of science tells us that a model should not be assessed based on the reasonableness of its assumptions or the sophistication of its analytics. It should be assessed based on the usefulness of its predictions. Backtesting is a process of assessing the usefulness of a value-at-risk measure’s predictions when applied to a particular portfolio over time."

Quote from: Value-at-Risk: Theory and Practice, 2nd Edition by Glyn A. Holton, published online in 2013.

## (2) What claim are we even testing?
We are testing the claims we made in notebook 3 about VaR, e.g. at 95% a one-day VaR forecast claims that the next realized loss should exceed its forecast threshold about 5% of the time; at 99%, about 1%, and so on.

## (3) What counts as evidence?
Each forecast made in the backtest MUST use only data/information available up to and including the forecast date. Only then can we compared that to the next actual price change in the historical data, which then produces a dated sequence of violations.

## (4) What tests?
We choose a Kupiec Kupiec Proportion of Failures (POF) test and a Christoffersen independence test as both are wildly used in financial engineering/risk management for backtesting VaR/ES, and because they test two distinct, necessary properties of VaR forecasts: breach frequency and one-lag independence.

### (4.1) Why these two in particular?
They together cover what we need to deduce the usefulness of our risk measures: Kupiec asks whether the overall number of violations is consistent with the claimed rate, and Christoffersen’s independence test asks whether a violation yesterday changes the chance of one today.

## (5) What is outside the scope of this backtest?
These tests assess our VaR thresholds, not our ES forecasts, and neither test proves the entire loss distribution is correct. ES concerns the severity of tail losses, not simply how many times a threshold was crossed.

# Derivation of Kupiec's Proportion of Failures (POF) Test

## Preliminary Calculations and Assumptions

**Assumptions**:
- We define a violation indicator $I_t = \mathbb{1}_{L_t > VaR_\alpha}$ for each day $t$ in the backtesting window, where $L_t$ is the realized loss and $VaR_\alpha$ is the model's VaR forecast for that day.
- We assume $I_1, I_2, \dots, I_n$ are iid Bernoulli random variables under the null hypothesis that the model is correctly calibrated, i.e. $I_t \sim \text{Bernoulli}(p)$ with true violation probability $p = 1-\alpha$.
- Let $n_1 = \sum_{t=1}^n I_t$ denote the number of violations, $n_0 = n - n_1$ the number of non-violations, and $n = n_0+n_1$ the total number of observations.

**Setup**:
The joint likelihood of $I_1,\dots,I_n$ given $p$ is
$$L(p) = p^{n_1}(1-p)^{n_0}\quad(1)$$

We define two parameter spaces:
- **Null hypothesis** $H_0: p \in \Theta_0 = \{1-\alpha\}$, a single fixed point (the model's claimed violation rate), so $r_0 = 0$ free parameters.
- **Unrestricted (alternative) hypothesis**: $p \in \Theta = (0,1)$, so $r=1$ free parameter.

## Deriving the MLE Under $\Theta$

We maximize the log-likelihood of (1) over $p\in(0,1)$. Taking the log,
$$\ell(p) = n_1\ln p + n_0\ln(1-p)$$

Differentiating and setting to zero,
$$\frac{d\ell}{dp} = \frac{n_1}{p} - \frac{n_0}{1-p} = 0$$

Solving,
$$n_1(1-p) = n_0 p \implies n_1 = p(n_0+n_1) = pn$$

$$\hat p = \frac{n_1}{n}\quad(2)$$

which is the observed violation rate — the natural estimator.

## Building the Likelihood Ratio

Since $\Theta_0$ is a single point, the maximum of $L(p)$ over $\Theta_0$ is just $L(1-\alpha)$ evaluated directly. The maximum over $\Theta$ is $L(\hat p)$ using (2). The likelihood ratio is
$$\lambda = \frac{\max_{p\in\Theta_0}L(p)}{\max_{p\in\Theta}L(p)} = \frac{L(1-\alpha)}{L(\hat p)} = \frac{(1-\alpha)^{n_1}\alpha^{n_0}}{\hat p^{n_1}(1-\hat p)^{n_0}}\quad(3)$$

## Applying the Asymptotic Theorem

By the likelihood ratio theorem (Wilks' theorem), since $\Theta$ has $r=1$ free parameter and $\Theta_0$ has $r_0=0$, the test statistic $-2\ln\lambda$ is asymptotically $\chi^2$ with $r-r_0 = 1$ degree of freedom under $H_0$.

Taking $-2\ln$ of (3):
$$LR_{POF} = -2\left[n_1\ln(1-\alpha) + n_0\ln(\alpha) - n_1\ln\hat p - n_0\ln(1-\hat p)\right]$$

$$\boxed{LR_{POF} = -2\ln\left[\frac{(1-\alpha)^{n_1}\alpha^{n_0}}{\hat p^{n_1}(1-\hat p)^{n_0}}\right]\sim\chi^2_{(1)}}\quad(4)$$

We reject the null hypothesis (i.e. conclude the VaR model is miscalibrated) if $LR_{POF}$ exceeds the $\chi^2_{(1)}$ critical value at our chosen test size (e.g. $3.84$ at 5% significance).

# Derivation of Christoffersen's Independence Test

## Preliminary Calculations and Assumptions

**Assumptions**:
- We reuse the violation indicator $I_t = \mathbb{1}_{L_t > VaR_\alpha}$ from the Kupiec test.
- We now model $\{I_t\}$ as a first-order Markov chain, so the probability of a violation on day $t$ is allowed to depend on whether day $t-1$ was a violation:
$$\pi_0 = \mathbb{P}(I_t=1 \mid I_{t-1}=0), \qquad \pi_1 = \mathbb{P}(I_t=1\mid I_{t-1}=1)$$
- Let $n_{ij}$ denote the number of days where $I_{t-1}=i$ and $I_t=j$, for $i,j\in\{0,1\}$. Note $n_{i0}+n_{i1}$ is the total number of days following a day in state $i$.
- **Intuition**: if the model is well-calibrated and violations are genuinely independent events, then knowing yesterday's outcome should tell us nothing about today's violation probability, i.e. $\pi_0=\pi_1$. If instead violations cluster — a string of losses breaching VaR back-to-back during a crisis — that means $\pi_1 \gg \pi_0$, revealing a blind spot: the model's assumptions (e.g. constant volatility) break down precisely when volatility regimes shift, and this test is designed to catch exactly that failure mode.

## Setting Up the Likelihoods

The likelihood of the observed transition counts, treating each row of the transition matrix as its own Bernoulli process, is
$$L(\pi_0,\pi_1) = (1-\pi_0)^{n_{00}}\pi_0^{n_{01}}(1-\pi_1)^{n_{10}}\pi_1^{n_{11}}\quad(1)$$

We define two parameter spaces:
- **Unrestricted (alternative) hypothesis**: $(\pi_0,\pi_1)\in\Theta = (0,1)^2$, so $r=2$ free parameters.
- **Null hypothesis** $H_0: \pi_0=\pi_1=\pi$ (independence), so $(\pi_0,\pi_1)\in\Theta_0$ collapses to a single free parameter $\pi\in(0,1)$, giving $r_0=1$.

## Deriving the MLEs Under $\Theta$

Notice (1) factors into two separate pieces, one depending only on $\pi_0$ and one only on $\pi_1$. We can therefore maximize each independently, exactly as we did for Kupiec's single Bernoulli likelihood.

Taking the log of the $\pi_0$-piece,
$$\ell(\pi_0) = n_{01}\ln\pi_0 + n_{00}\ln(1-\pi_0)$$

Differentiating and setting to zero,
$$\frac{d\ell}{d\pi_0} = \frac{n_{01}}{\pi_0} - \frac{n_{00}}{1-\pi_0} = 0 \implies \hat\pi_0 = \frac{n_{01}}{n_{00}+n_{01}}\quad(2)$$

By an identical argument on the $\pi_1$-piece,
$$\hat\pi_1 = \frac{n_{11}}{n_{10}+n_{11}}\quad(3)$$

Both are just the observed violation rate conditional on yesterday's state — the natural estimators.

## Deriving the MLE Under $\Theta_0$

Under $H_0$, (1) collapses to
$$L(\pi) = (1-\pi)^{n_{00}+n_{10}}\pi^{n_{01}+n_{11}}$$

which is structurally identical to the single Bernoulli likelihood from Kupiec's derivation. By the same maximization,
$$\hat\pi = \frac{n_{01}+n_{11}}{n_{00}+n_{01}+n_{10}+n_{11}} = \frac{n_{01}+n_{11}}{n}\quad(4)$$

## Building the Likelihood Ratio

$$\lambda = \frac{\max_{\pi\in\Theta_0}L(\pi)}{\max_{(\pi_0,\pi_1)\in\Theta}L(\pi_0,\pi_1)} = \frac{(1-\hat\pi)^{n_{00}+n_{10}}\hat\pi^{n_{01}+n_{11}}}{(1-\hat\pi_0)^{n_{00}}\hat\pi_0^{n_{01}}(1-\hat\pi_1)^{n_{10}}\hat\pi_1^{n_{11}}}\quad(5)$$

## Applying the Asymptotic Theorem

By the same likelihood ratio theorem used for Kupiec, since $\Theta$ has $r=2$ free parameters and $\Theta_0$ has $r_0=1$, the statistic $-2\ln\lambda$ is asymptotically $\chi^2$ with $r-r_0=1$ degree of freedom under $H_0$.

$$LR_{ind} = -2\left[(n_{00}+n_{10})\ln(1-\hat\pi) + (n_{01}+n_{11})\ln\hat\pi - n_{00}\ln(1-\hat\pi_0) - n_{01}\ln\hat\pi_0 - n_{10}ln(1-\hat\pi_1) - n_{11}ln\hat\pi_1\right]$$
$$\boxed{LR_{ind} = -2\ln\left[\frac{(1-\hat\pi)^{n_{00}+n_{10}}\hat\pi^{n_{01}+n_{11}}}{(1-\hat\pi_0)^{n_{00}}\hat\pi_0^{n_{01}}(1-\hat\pi_1)^{n_{10}}\hat\pi_1^{n_{11}}}\right]\sim\chi^2_{(1)}}\quad(6)$$

We reject the null hypothesis of independence (i.e. conclude violations cluster, signaling a blind spot where the model's assumptions break down under stress) if $LR_{ind}$ exceeds the $\chi^2_{(1)}$ critical value at our chosen significance level.

Cross-referenced with Christoffersen, P. (1998), "Evaluating Interval Forecasts," *International Economic Review*, 39(4), pp. 841-862.

In [11]:
# for modules to work when running this notebook, we need to add the project root to the sys.path
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [12]:
# imports
import numpy as np
import pandas as pd
import numpy.typing as npt
import scipy.stats as stats

from data.data import get_multiple_stocks_data

In [13]:
class BacktestingEngine:
    ROLLING_PERIODS = (20, 60)
    TIME_HORIZONS = (1, 10)
    ALPHAS = (0.95, 0.99)  # the confidence levels for VaR and ES calculations
    DAILY = 252
    N_PATHS = 10_000
    LAMBDA_ = 0.94
    T = 1.0    # Time to maturity in years for the simulation
    N = 252    # Number of time steps in the simulation (daily steps for 1 year)

    def get_conf(self, alpha: float) -> int:
        if alpha not in self.ALPHAS:
            raise ValueError(f"Alpha must be one of {self.ALPHAS}")
        return int(alpha * 100)

    def ewma_volatility(self, log_returns: pd.Series, lambda_: float = LAMBDA_) -> float:
        if not 0 < lambda_ <= 1:
            raise ValueError("Lambda must be between 0 and 1")

        squared_returns = log_returns ** 2
        last_return = squared_returns.iloc[-1]
        squared_returns = squared_returns.shift(1)
        squared_returns = squared_returns.fillna(value=log_returns.mean() ** 2)
        ewma_var = squared_returns.ewm(alpha=1 - lambda_, adjust=False).mean().iloc[-1]

        forecast = (last_return * (1 - lambda_) + ewma_var * lambda_) ** 0.5 * np.sqrt(self.DAILY)
        return forecast

    def simulate_gbm(
        self,
        S0: float,
        mu_annual: float,
        sigma_annual: float,
        T: float,
        N: int,
        n_paths: int
    ) -> npt.NDArray[np.float64]:
        N = int(N)
        dt = T / N
        z = np.random.standard_normal((n_paths, N))
        log_increments = (mu_annual - 0.5 * sigma_annual ** 2) * dt + sigma_annual * np.sqrt(dt) * z
        paths = np.zeros((n_paths, N + 1))
        paths[:, 0] = S0
        paths[:, 1:] = S0 * np.exp(np.cumsum(log_increments, axis=1))
        return paths

    # --- Backtesting statistical tests (verified correct against independent manual re-derivation) ---

    def kupiec_pof_test(
        self,
        violations: int,
        observations: int,
        alpha: float
    ) -> tuple[float, float]:
        if observations <= 0:
            raise ValueError("Number of observations must be positive")
        if not (0 < alpha < 1):
            raise ValueError("Alpha must be between 0 and 1")

        non_violations = observations - violations  # n - n_1, i.e. n_0
        p_hat = violations / observations

        if p_hat == 0 or p_hat == 1:
            # log(0) is undefined; by convention, treat the null-model likelihood
            # ratio as the limiting case since n_1*ln(p_hat) -> 0 as p_hat -> 0.
            likelihood_ratio = -2 * (
                violations * np.log(1 - alpha) + non_violations * np.log(alpha)
            )
        else:
            likelihood_ratio = -2 * (
                violations * np.log(1 - alpha) + non_violations * np.log(alpha)
                - violations * np.log(p_hat) - non_violations * np.log(1 - p_hat)
            )

        # P-value based on Chi-square distribution with 1 degree of freedom
        p_value = 1 - stats.chi2.cdf(likelihood_ratio, df=1)
        return float(likelihood_ratio), float(p_value)

    def christoffersen_independence_test(
        self,
        violations: npt.NDArray[np.int64]
    ) -> tuple[float, float]:
        if len(violations) < 2:
            raise ValueError("At least two observations are required for the Christoffersen test.")

        # n_ij denotes the number of days where I_{t-1}=i and I_t=j (I_t is the
        # violation indicator, 1=violation, 0=no violation), for i,j in {0,1}.
        n00 = np.sum((violations[:-1] == 0) & (violations[1:] == 0))
        n01 = np.sum((violations[:-1] == 0) & (violations[1:] == 1))
        n10 = np.sum((violations[:-1] == 1) & (violations[1:] == 0))
        n11 = np.sum((violations[:-1] == 1) & (violations[1:] == 1))

        n0 = n00 + n01
        n1 = n10 + n11

        if n0 == 0 or n1 == 0:
            # No transitions observed out of one of the states -- test undefined.
            return float("nan"), float("nan")

        pi0_hat = n01 / n0
        pi1_hat = n11 / n1
        pi_hat = (n01 + n11) / (n0 + n1)

        def safe_term(count, prob):
            return count * np.log(prob) if count > 0 else 0.0

        likelihood_ratio = -2 * (
            safe_term(n00 + n10, 1 - pi_hat) + safe_term(n01 + n11, pi_hat)
            - safe_term(n00, 1 - pi0_hat) - safe_term(n01, pi0_hat)
            - safe_term(n10, 1 - pi1_hat) - safe_term(n11, pi1_hat)
        )

        p_value = 1 - stats.chi2.cdf(likelihood_ratio, df=1)
        return float(likelihood_ratio), float(p_value)

    # --- The actual backtest: walk-forward over real historical data, no lookahead ---

    def walk_forward_backtest(
        self,
        log_returns: pd.Series,
        P0_series: pd.Series,
        alpha: float,
        burn_in: int = 100,
        lambda_: float = LAMBDA_
    ) -> dict[str, npt.NDArray]:
        """
        Walk-forward backtesting of 1-day parametric VaR forecasts using EWMA volatility.
        At each day t, only data up to and including t is used to forecast day t+1 --
        no lookahead. Returns violations (0/1 series), var_series, and dates.
        """
        n = len(log_returns)
        violations = []
        var_series = []

        for t in range(burn_in, n - 1):
            history = log_returns.iloc[:t + 1]

            mu_annual = history.mean() * self.DAILY
            sigma_annual = self.ewma_volatility(history, lambda_=lambda_)

            P0 = P0_series.iloc[t + 1]  # # price known when the forecast is made, accounts for log return at t being price[t+1] / price[t]
            delta_t = 1 / self.DAILY  # 1-day horizon
            z_alpha = stats.norm.ppf(alpha)

            adjusted_mu = delta_t * (mu_annual - 0.5 * sigma_annual ** 2)
            adjusted_sigma = sigma_annual * np.sqrt(delta_t)
            var = -P0 * (adjusted_mu - adjusted_sigma * z_alpha)

            # actual realized loss on the next observed price
            P1 = P0_series.iloc[t + 2]
            realized_loss = P0 - P1

            violations.append(int(realized_loss > var))
            var_series.append(var)

        return {
            "violations": np.array(violations, dtype=int),
            "var_series": np.array(var_series),
            "dates": log_returns.index[burn_in + 1:n]
        }

In [14]:
# ----------------------------------------------------------------------------
# Data pipeline
# ----------------------------------------------------------------------------
backtesting_eng = BacktestingEngine()
tickers = ["SPY", "GLD", "NVDA", "GOOGL", "BTC-USD"]
data = get_multiple_stocks_data(tickers, "2016-06-01", "2026-06-01", "1d")

prices = {}
log_returns = {}
paths_by_ticker = {}
backtest_results = {} # walk-forward backtest: backtest_results[ticker][f"conf_{conf}"]

for ticker in tickers:
    curr_data = data[ticker]
    prices[ticker] = curr_data["Close"]

    lr = pd.Series(
        np.diff(np.log(prices[ticker].values)),
        index=prices[ticker].index[1:]
    )
    log_returns[ticker] = lr

    # --- Backtest: ONE walk-forward result per (ticker, alpha) ---
    backtest_results[ticker] = {}
    for alpha in backtesting_eng.ALPHAS:
        conf = backtesting_eng.get_conf(alpha)
        result = backtesting_eng.walk_forward_backtest(
            log_returns[ticker], prices[ticker], alpha=alpha, burn_in=100
        )

        kupiec_lr, kupiec_p = backtesting_eng.kupiec_pof_test(
            violations=int(result["violations"].sum()),
            observations=len(result["violations"]),
            alpha=alpha
        )
        christoffersen_lr, christoffersen_p = backtesting_eng.christoffersen_independence_test(
            result["violations"]
        )

        n_obs = len(result["violations"])
        n_viol = int(result["violations"].sum())

        backtest_results[ticker][f"conf_{conf}"] = {
            "n_obs": n_obs,
            "violations": n_viol,
            "violation_rate": n_viol / n_obs,
            "target_rate": 1 - alpha,
            "kupiec_lr": kupiec_lr,
            "kupiec_p": kupiec_p,
            "christoffersen_lr": christoffersen_lr,
            "christoffersen_p": christoffersen_p,
        }

In [15]:
# ----------------------------------------------------------------------------
# Table 1: Walk-forward backtest results (Kupiec POF + Christoffersen independence)
# ----------------------------------------------------------------------------
bt_rows = []
for ticker in tickers:
    for conf_key, stats_dict in backtest_results[ticker].items():
        conf = conf_key.replace("conf_", "")
        bt_rows.append({
            "Ticker": ticker,
            "Confidence": f"{conf}%",
            "Observations": stats_dict["n_obs"],
            "Violations": stats_dict["violations"],
            "Violation Rate": stats_dict["violation_rate"],
            "Target Rate": stats_dict["target_rate"],
            "Kupiec LR": stats_dict["kupiec_lr"],
            "Kupiec p-value": stats_dict["kupiec_p"],
            "Christoffersen LR": stats_dict["christoffersen_lr"],
            "Christoffersen p-value": stats_dict["christoffersen_p"],
        })

backtest_table = pd.DataFrame(bt_rows).sort_values(["Ticker", "Confidence"]).reset_index(drop=True)


def highlight_reject(val):
    """Flags p-values below 0.05 (reject H0 at 5% significance) in red; pass in blue."""
    if pd.isna(val):
        return "background-color: #888888; color: white;"
    return "background-color: #b40426; color: white;" if val < 0.05 else "background-color: #3b4cc0; color: white;"


styled_backtest = (
    backtest_table.style
    .format({
        "Violation Rate": "{:.4f}",
        "Target Rate": "{:.4f}",
        "Kupiec LR": "{:.4f}",
        "Kupiec p-value": "{:.4f}",
        "Christoffersen LR": "{:.4f}",
        "Christoffersen p-value": "{:.4f}",
    })
    .map(highlight_reject, subset=["Kupiec p-value", "Christoffersen p-value"])
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("background-color", "#2c3e50"),
                                       ("color", "white"), ("font-weight", "bold"), ("padding", "6px 10px")]},
        {"selector": "td", "props": [("padding", "6px 10px")]},
        {"selector": "table", "props": [("border-collapse", "collapse"),
                                          ("font-family", "Arial, sans-serif"), ("font-size", "13px")]},
        {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold"), ("padding", "8px 0")]},
         # Hover for plain (non-gradient) cells: simple background tint.
        {"selector": "tbody tr", "props": [("transition", "background-color 0.3s ease")]},
        {"selector": "tbody tr:hover", "props": [("background-color", "#494a4b")]},
        # Hover overlay for gradient cells: an inset box-shadow layers a
        # translucent dark tint ON TOP of the inline background-color that
        # background_gradient() sets, since background-color itself can't
        # be reliably overridden by a plain CSS hover rule once it's inline.
        {"selector": "tbody tr td", "props": [("transition", "box-shadow 0.3s ease")]},
        {"selector": "tbody tr:hover td", "props": [
            ("box-shadow", "inset 0 0 0 9999px rgba(0,0,0,0.12)")
        ]},
    ])
    .set_caption("Walk-Forward Backtest: Kupiec POF & Christoffersen Independence Tests (red = reject H0)")
)

styled_backtest

,Ticker,Confidence,Observations,Violations,Violation Rate,Target Rate,Kupiec LR,Kupiec p-value,Christoffersen LR,Christoffersen p-value
0,BTC-USD,95%,3550,180,0.0507,0.0500,0.0369,0.8477,2.5796,0.1082
1,BTC-USD,99%,3550,75,0.0211,0.0100,33.6389,0.0000,2.7382,0.0980
2,GLD,95%,2411,126,0.0523,0.0500,0.2557,0.6131,4.0803,0.0434
3,GLD,99%,2411,46,0.0191,0.0100,15.8547,0.0001,0.0169,0.8966
4,GOOGL,95%,2411,122,0.0506,0.0500,0.0183,0.8924,0.0056,0.9403
5,GOOGL,99%,2411,53,0.0220,0.0100,26.0636,0.0000,0.0258,0.8724
6,NVDA,95%,2411,130,0.0539,0.0500,0.7612,0.3830,4.6830,0.0305
7,NVDA,99%,2411,38,0.0158,0.0100,6.8779,0.0087,1.2176,0.2698
8,SPY,95%,2411,141,0.0585,0.0500,3.4712,0.0624,1.7137,0.1905
9,SPY,99%,2411,58,0.0241,0.0100,34.5302,0.0000,3.4604,0.0629


# Summary Analysis of Results
Interestingly enough, we have only 3 cases as no asset had enough evidence to reject the respective $H_0$ in both tests:
  (i) Kupiec rejects, Christoffersen fails to reject
  (ii) Kupiec fails to reject, Christoffersen rejects
  (iii) Both test fail to reject their respective $H_0$

Let's explore case by case:

## (i) Kupiec rejects, Christoffersen fails to reject
This interestingly enough happened for all of our assets at the 99% confidence level, which indicates at that level our VaR model is much too leniant in its risk measure (i.e. the 99% VaR thresholds understate tail risk), but at different degrees for different assets. For example, NVDA has a 1.58% violation/breach rate (expected 1%) compared to SPY's 2.41%, so not all assets have as severe of deviations from the expected violation rate. Because Christoffersen fails to reject, we can't say there is enough evidence that there is one-lag dependence in our breaches. 

## (ii) Kupiec fails to reject, Christoffersen rejects
This only happened for 2 assets, NVDA and GLD, at the 95% confidence level. In general, it means that frequency looks statistically compatible, but the timing does not look independent under this test. For these particular assets, it makes sense (from a purely speculative, macroeconomic lens, more in depth data analysis would be necessary to make stronger, more backed conclusions), as they are both heavily tied to the macroeconomic environment, for example gold historically performs better in a recession, and vice-versa, so its risk in downtimes will cluster into one time period. Similarly for NVDA, but in a good macro environment.

## (iii) Both test fail to reject
This means that no failure was detected by these two tests at a 5% significance level, meaning we don't have enough evidence against the frequency of risk specified by our model being wrong, nor do we have evidence against risk breaches being one-lag independent. Overall, for these asset–confidence-level combinations, the backtest did not detect a problem in breach frequency or one-lag independence. This is encouraging, but it is not enough on its own to establish that the model is reliable for risk-management decisions.